In [2]:
# Installation des dépendances
!pip install -q transformers accelerate peft bitsandbytes
!pip install -q scikit-learn pandas matplotlib seaborn
!pip install -q sentencepiece protobuf tqdm

# Vérifier le GPU
import torch
import os

print("🔥 Vérification GPU:")
print(f"✅ GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ Mémoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("❌ ERREUR: Pas de GPU détecté !")

🔥 Vérification GPU:
✅ GPU disponible: True
✅ GPU: Tesla T4
✅ Mémoire GPU: 15.83 GB


In [3]:
import os
import pandas as pd
from PIL import Image
import re

# Créer les dossiers nécessaires
os.makedirs('/kaggle/working/results', exist_ok=True)

# ✅ CHEMIN CORRIGÉ - Double "aligned"
IMG_DIR = '/kaggle/input/data-cv/aligned/aligned'
LABEL_FILE = '/kaggle/input/data-cv/RAFCE_emolabel (1).txt'
PARTITION_FILE = '/kaggle/input/data-cv/RAFCE_partition (1).txt'

# Vérifier que les fichiers existent
print("📂 Vérification des fichiers:")
print(f"✅ Images dir: {os.path.exists(IMG_DIR)}")
print(f"✅ Labels: {os.path.exists(LABEL_FILE)}")
print(f"✅ Partition: {os.path.exists(PARTITION_FILE)}")

# Mapping des émotions
EMOTION_MAP = {
    0: "Happily Surprised",
    1: "Happily Disgusted",
    2: "Sadly Fearful",
    3: "Sadly Angry",
    4: "Sadly Surprised",
    5: "Sadly Disgusted",
    6: "Fearfully Angry",
    7: "Fearfully Surprised",
    8: "Fearfully Disgusted",
    9: "Angry Surprised",
    10: "Angry Disgusted",
    11: "Disgustedly Surprised",
    12: "Neutral / Other Compound",
    13: "Neutral / Other Compound"
}

print(f"\n📋 {len(EMOTION_MAP)} émotions définies")

# Créer le mapping des fichiers
file_mapping = {}
all_files = os.listdir(IMG_DIR)

print(f"\n🔍 Analyse de {len(all_files)} fichiers...")

for filename in all_files:
    # Accepter toutes les images
    if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
        # Extraire tous les nombres du nom de fichier
        numbers = re.findall(r'(\d+)', filename)
        if numbers:
            number = numbers[0]
            
            # Créer plusieurs variantes de clés
            file_mapping[f"{number}.jpg"] = filename
            file_mapping[f"{int(number):04d}.jpg"] = filename
            file_mapping[f"{int(number):05d}.jpg"] = filename

unique_files = len(set(file_mapping.values()))
print(f"✅ {unique_files} fichiers uniques mappés")
print(f"   ({len(file_mapping)} entrées de mapping au total)")

# Afficher quelques exemples de mapping
print(f"\n📋 Exemples de mapping:")
for i, (key, val) in enumerate(list(file_mapping.items())[:10]):
    print(f"   {key} → {val}")

# Charger les données
labels_df = pd.read_csv(LABEL_FILE, sep=' ', header=None, names=['image', 'label'])
partition_df = pd.read_csv(PARTITION_FILE, sep=' ', header=None, names=['image', 'partition'])

data = pd.merge(labels_df, partition_df, on='image')
train_data = data[data['partition'] == 1].reset_index(drop=True)
test_data = data[data['partition'] == 2].reset_index(drop=True)

print(f"\n📊 Dataset chargé:")
print(f"   Train: {len(train_data)} images")
print(f"   Test: {len(test_data)} images")

# Vérifier combien d'images peuvent être trouvées
test_found = 0
test_examples = []
for img_name in test_data['image'].head(10):
    real_file = file_mapping.get(img_name)
    if real_file:
        test_found += 1
        test_examples.append(f"{img_name} → {real_file}")

print(f"\n✅ Test de mapping: {test_found}/10 premières images trouvées")

if test_found > 0:
    print(f"\n📸 Exemples de mapping réussi:")
    for ex in test_examples[:5]:
        print(f"   {ex}")

if test_found == 0:
    print("\n⚠️ ATTENTION: Aucune image trouvée!")
    print("Vérifiez le format des noms dans les fichiers .txt vs le dossier")

📂 Vérification des fichiers:
✅ Images dir: True
✅ Labels: True
✅ Partition: True

📋 14 émotions définies

🔍 Analyse de 4908 fichiers...
✅ 4908 fichiers uniques mappés
   (9816 entrées de mapping au total)

📋 Exemples de mapping:
   0661.jpg → 0661_aligned.jpg
   00661.jpg → 0661_aligned.jpg
   1964.jpg → 1964_aligned.jpg
   01964.jpg → 1964_aligned.jpg
   1793.jpg → 1793_aligned.jpg
   01793.jpg → 1793_aligned.jpg
   4135.jpg → 4135_aligned.jpg
   04135.jpg → 4135_aligned.jpg
   1118.jpg → 1118_aligned.jpg
   01118.jpg → 1118_aligned.jpg

📊 Dataset chargé:
   Train: 909 images
   Test: 931 images

✅ Test de mapping: 10/10 premières images trouvées

📸 Exemples de mapping réussi:
   0002.jpg → 0002_aligned.jpg
   0009.jpg → 0009_aligned.jpg
   0010.jpg → 0010_aligned.jpg
   0013.jpg → 0013_aligned.jpg
   0017.jpg → 0017_aligned.jpg


In [4]:
import os

print("🔍 DIAGNOSTIC DU DOSSIER ALIGNED")
print("="*70)

IMG_DIR = '/kaggle/input/data-cv/aligned'

# Vérifier le contenu
if os.path.exists(IMG_DIR):
    all_files = os.listdir(IMG_DIR)
    print(f"📂 Nombre total de fichiers: {len(all_files)}")
    
    # Afficher les 20 premiers
    print(f"\n📄 Exemples de fichiers (20 premiers):")
    for i, f in enumerate(all_files[:20]):
        print(f"   {i+1}. {f}")
    
    # Compter par extension
    extensions = {}
    for f in all_files:
        ext = os.path.splitext(f)[1].lower()
        extensions[ext] = extensions.get(ext, 0) + 1
    
    print(f"\n📊 Fichiers par extension:")
    for ext, count in sorted(extensions.items()):
        print(f"   {ext if ext else '(pas d\'extension)'}: {count} fichiers")
else:
    print(f"❌ Le dossier n'existe pas: {IMG_DIR}")

# Vérifier aussi la structure
print(f"\n📁 Structure de /kaggle/input/data-cv/:")
for item in os.listdir('/kaggle/input/data-cv/'):
    full_path = os.path.join('/kaggle/input/data-cv/', item)
    if os.path.isdir(full_path):
        num_files = len(os.listdir(full_path))
        print(f"   📁 {item}/ ({num_files} fichiers)")
    else:
        print(f"   📄 {item}")

🔍 DIAGNOSTIC DU DOSSIER ALIGNED
📂 Nombre total de fichiers: 1

📄 Exemples de fichiers (20 premiers):
   1. aligned

📊 Fichiers par extension:
   (pas d'extension): 1 fichiers

📁 Structure de /kaggle/input/data-cv/:
   📄 RAFCE_emolabel (1).txt
   📁 aligned/ (1 fichiers)
   📄 RAFCE_partition (1).txt


In [5]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import torch

print("🔄 Chargement de BLIP-2...")

# Charger le modèle
processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    load_in_8bit=True,  # Quantization pour économiser la mémoire
    device_map="auto",
    torch_dtype=torch.float16
)

print("✅ BLIP-2 chargé avec succès !")
print(f"📊 Paramètres totaux: {sum(p.numel() for p in model.parameters()):,}")

2026-01-19 09:40:12.867104: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768815613.126001     269 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768815613.200318     269 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768815613.773333     269 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768815613.773364     269 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768815613.773366     269 computation_placer.cc:177] computation placer alr

🔄 Chargement de BLIP-2...


`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(


✅ BLIP-2 chargé avec succès !
📊 Paramètres totaux: 3,744,761,856


In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("⚙️ Configuration de LoRA...")

# Préparer le modèle pour l'entraînement
model = prepare_model_for_kbit_training(model)

# Configuration LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Appliquer LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\n✅ LoRA configuré avec succès !")

⚙️ Configuration de LoRA...
trainable params: 2,621,440 || all params: 3,747,383,296 || trainable%: 0.0700

✅ LoRA configuré avec succès !


In [8]:
from torch.utils.data import Dataset
import torch

class RAFCETrainingDataset(Dataset):
    def __init__(self, data, img_dir, file_mapping, processor, emotion_map):
        self.data = data
        self.img_dir = img_dir
        self.file_mapping = file_mapping
        self.processor = processor
        self.emotion_map = emotion_map
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_name = row['image']
        label = row['label']
        
        real_filename = self.file_mapping.get(img_name)
        if not real_filename:
            return None
        
        img_path = os.path.join(self.img_dir, real_filename)
        
        try:
            image = Image.open(img_path).convert('RGB')
            emotion_name = self.emotion_map[label]
            
            prompt = "Question: What emotion is shown in this face? Answer:"
            answer = emotion_name
            
            encoding = self.processor(
                images=image,
                text=prompt,
                return_tensors="pt"
            )
            
            labels = self.processor.tokenizer(
                answer,
                return_tensors="pt"
            ).input_ids
            
            return {
                'pixel_values': encoding.pixel_values.squeeze(),
                'input_ids': encoding.input_ids.squeeze(),
                'labels': labels.squeeze()
            }
        except:
            return None

# ✅ RÉDUIRE À 500 IMAGES pour être plus rapide
train_subset = train_data.sample(500, random_state=42)
train_dataset = RAFCETrainingDataset(
    train_subset,
    IMG_DIR,
    file_mapping,
    processor,
    EMOTION_MAP
)

print(f"✅ Dataset créé: {len(train_dataset)} images")
print(f"⏱️ Temps estimé: ~30 minutes pour 1 epoch")

✅ Dataset créé: 500 images
⏱️ Temps estimé: ~30 minutes pour 1 epoch


In [ ]:
from transformers import Trainer, TrainingArguments

print("🚀 FINE-TUNING - 3 EPOCHS")
print("="*70)
print("⏱️ Temps estimé: 6-8 minutes")
print("🎯 Objectif: Atteindre 55-65% accuracy\n")

# Arguments - 3 EPOCHS (CORRIGÉ)
training_args = TrainingArguments(
    output_dir="/kaggle/working/blip2_finetuned_3epochs",
    num_train_epochs=3,  # ✅ 3 EPOCHS
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    remove_unused_columns=False,
    report_to="none",
    # ❌ RETIRÉ: evaluation_strategy="no"
)

# Collator
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.nn.utils.rnn.pad_sequence(
            [b['input_ids'] for b in batch],
            batch_first=True,
            padding_value=processor.tokenizer.pad_token_id
        ),
        'labels': torch.nn.utils.rnn.pad_sequence(
            [b['labels'] for b in batch],
            batch_first=True,
            padding_value=-100
        )
    }

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)

# LANCER
print("🔄 Démarrage de l'entraînement...\n")
trainer.train()

print("\n" + "="*70)
print("✅ FINE-TUNING TERMINÉ (3 EPOCHS)")
print("="*70)
print("📊 Le modèle devrait maintenant avoir une accuracy ~55-65%")


In [ ]:
from tqdm import tqdm

print("🧪 Évaluation sur 50 images du test set")
print("="*70)

# Golden Prompt
LABEL_LIST_STR = ", ".join([f"{k}: {v}" for k, v in EMOTION_MAP.items()])
golden_prompt = f"""Analyze the facial expression. Identify the compound emotion from this list: {LABEL_LIST_STR}.
Answer with only the emotion name."""

# Fonction de prédiction
def predict_emotion(image_path, prompt):
    try:
        image = Image.open(image_path).convert('RGB')
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, torch.float16)
        
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        
        # Parser
        emotion_id = -1
        for idx, label in EMOTION_MAP.items():
            if label.lower() in generated_text.lower():
                emotion_id = idx
                break
        
        return generated_text, emotion_id, EMOTION_MAP.get(emotion_id, "Unknown")
    except:
        return None, -1, None

# Tester sur 50 images
test_sample = test_data.head(50)
results = []

for idx, row in tqdm(test_sample.iterrows(), total=len(test_sample)):
    img_name = row['image']
    true_label = row['label']
    
    real_filename = file_mapping.get(img_name)
    if not real_filename:
        continue
    
    img_path = f'{IMG_DIR}/{real_filename}'
    if not os.path.exists(img_path):
        continue
    
    full_text, pred_id, emotion_name = predict_emotion(img_path, golden_prompt)
    
    results.append({
        'ImageID': img_name,
        'True_Label': true_label,
        'True_Emotion': EMOTION_MAP[true_label],
        'Pred_ID': pred_id,
        'Pred_Emotion': emotion_name
    })

# Calculer accuracy
df_results = pd.DataFrame(results)
valid_preds = df_results[df_results['Pred_ID'] != -1]

if len(valid_preds) > 0:
    from sklearn.metrics import accuracy_score, f1_score
    
    accuracy = accuracy_score(valid_preds['True_Label'], valid_preds['Pred_ID'])
    f1 = f1_score(valid_preds['True_Label'], valid_preds['Pred_ID'], average='macro')
    
    print(f"\n📊 RÉSULTATS (sur 50 images):")
    print(f"✅ Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"✅ F1-Macro: {f1:.4f}")
    print(f"✅ Prédictions valides: {len(valid_preds)}/{len(df_results)}")
    
    df_results.to_csv('/kaggle/working/results/resultats_blip2.csv', index=False)

In [ ]:
print("🔍 DIAGNOSTIC - Exemples de réponses du modèle")
print("="*70)

# Tester sur 5 images
test_sample = test_data.head(5)

for i, (_, row) in enumerate(test_sample.iterrows()):
    img_name = row['image']
    true_label = row['label']
    true_emotion = EMOTION_MAP[true_label]
    
    real_file = file_mapping.get(img_name)
    if not real_file:
        continue
    
    img_path = f"{IMG_DIR}/{real_file}"
    
    try:
        image = Image.open(img_path).convert('RGB')
        
        # Prompt simple
        prompt = "Question: What emotion is shown in this face? Answer:"
        
        inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, torch.float16)
        generated_ids = model.generate(**inputs, max_new_tokens=50)
        response = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        
        print(f"\n{'='*70}")
        print(f"📸 Image {i+1}: {img_name}")
        print(f"✅ Vérité: {true_emotion}")
        print(f"🤖 Réponse du modèle:")
        print(f"   '{response}'")
        
    except Exception as e:
        print(f"❌ Erreur: {e}")

In [9]:
class FacialExpressionDataset(Dataset):
    def __init__(self, dataset, processor, file_mapping, img_dir):
        self.dataset = dataset
        self.processor = processor
        self.file_mapping = file_mapping
        self.img_dir = img_dir

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset.iloc[idx]
        img_name = item['image']
        label_idx = item['label']
        
        # --- 1. Gestion Image (inchangé) ---
        real_filename = self.file_mapping.get(img_name)
        if not real_filename:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        else:
            image_path = os.path.join(self.img_dir, real_filename)
            try:
                image = Image.open(image_path).convert('RGB')
            except:
                image = Image.new('RGB', (224, 224), (0, 0, 0))

        # --- 2. Prompt Engineering & Labels (CORRECTION ICI) ---
        emotion_text = EMOTION_MAP.get(label_idx, "Unknown")
        
        question = "Question: What is the facial expression? Answer:"
        full_text = f"{question} {emotion_text}"
        
        # On encode l'image et le texte complet
        inputs = self.processor(
            images=image,
            text=full_text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=64
        )
        
        # Pour un Causal LM (BLIP-2), les labels sont identiques aux input_ids
        input_ids = inputs["input_ids"]
        labels = input_ids.clone()
        
        # --- 3. Masquage (Masking) ---
        # On ne veut pas que le modèle apprenne à réciter la question.
        # On masque toute la partie "Question..." avec -100.
        
        # On calcule la longueur de la question seule en tokens
        # Note: On utilise le tokenizer directement pour la rapidité
        question_tokens = self.processor.tokenizer(question, return_attention_mask=False)["input_ids"]
        question_len = len(question_tokens)
        
        # Si le tokenizer ajoute un token de début (BOS), on ajuste (souvent -1 ou -2 selon le tokenizer)
        # Pour OPT/BLIP, une approximation conservatrice fonctionne bien :
        mask_len = min(question_len, labels.shape[1])
        
        # On applique le masque -100 (Ignored index) sur la question
        labels[:, :mask_len] = -100
        
        # On masque aussi le padding (pour ne pas apprendre le vide)
        labels[input_ids == self.processor.tokenizer.pad_token_id] = -100
        
        # On ajoute la clé 'labels' au dictionnaire
        inputs["labels"] = labels
        
        # On retire la dimension de batch (1, seq_len) -> (seq_len)
        return {k: v.squeeze(0) for k, v in inputs.items()}

In [12]:
import os
import torch
import pandas as pd
import re
from PIL import Image
from torch.utils.data import Dataset
from transformers import AutoProcessor, Blip2ForConditionalGeneration, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

print("🚀 DÉMARRAGE COMPLET (Tout-en-un)")
print("="*50)

# --- 1. CONFIGURATION & DONNÉES ---
IMG_DIR = '/kaggle/input/data-cv/aligned/aligned'
LABEL_FILE = '/kaggle/input/data-cv/RAFCE_emolabel (1).txt'
PARTITION_FILE = '/kaggle/input/data-cv/RAFCE_partition (1).txt'

EMOTION_MAP = {
    0: "Happily Surprised", 1: "Happily Disgusted", 2: "Sadly Fearful",
    3: "Sadly Angry", 4: "Sadly Surprised", 5: "Sadly Disgusted",
    6: "Fearfully Angry", 7: "Fearfully Surprised", 8: "Fearfully Disgusted",
    9: "Angry Surprised", 10: "Angry Disgusted", 11: "Disgustedly Surprised",
    12: "Neutral", 13: "Neutral"
}

# Mapping fichiers
print("📂 Mapping des fichiers...")
file_mapping = {}
if os.path.exists(IMG_DIR):
    for filename in os.listdir(IMG_DIR):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            numbers = re.findall(r'(\d+)', filename)
            if numbers:
                n = numbers[0]
                file_mapping[f"{n}.jpg"] = filename
                file_mapping[f"{int(n):04d}.jpg"] = filename
                file_mapping[f"{int(n):05d}.jpg"] = filename

# Chargement DataFrames
print("📊 Chargement des CSV...")
labels_df = pd.read_csv(LABEL_FILE, sep=' ', header=None, names=['image', 'label'])
partition_df = pd.read_csv(PARTITION_FILE, sep=' ', header=None, names=['image', 'partition'])
data = pd.merge(labels_df, partition_df, on='image')
train_df = data[data['partition'] == 1].reset_index(drop=True)
test_df = data[data['partition'] == 2].reset_index(drop=True)

# --- 2. MODÈLE & LoRA ---
print("🤖 Chargement du modèle BLIP-2...")
model_id = "Salesforce/blip2-opt-2.7b"
processor = AutoProcessor.from_pretrained(model_id)
model = Blip2ForConditionalGeneration.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.float16
)

# Configuration LoRA
peft_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# --- 3. DATASET CLASS (CORRIGÉE) ---
class FacialExpressionDataset(Dataset):
    def __init__(self, dataset, processor, file_mapping, img_dir):
        self.dataset = dataset
        self.processor = processor
        self.file_mapping = file_mapping
        self.img_dir = img_dir

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset.iloc[idx]
        real_file = self.file_mapping.get(item['image'])
        
        # Image
        if real_file:
            try:
                img_path = os.path.join(self.img_dir, real_file)
                image = Image.open(img_path).convert('RGB')
            except:
                image = Image.new('RGB', (224, 224), (0, 0, 0))
        else:
            image = Image.new('RGB', (224, 224), (0, 0, 0))

        # Prompt & Label
        emotion = EMOTION_MAP.get(item['label'], "Unknown")
        question = "Question: What is the facial expression? Answer:"
        text = f"{question} {emotion}"
        
        inputs = self.processor(images=image, text=text, return_tensors="pt", 
                                padding="max_length", truncation=True, max_length=64)
        
        input_ids = inputs["input_ids"].squeeze(0)
        labels = input_ids.clone()
        
        # Masking question (-100)
        q_tokens = self.processor.tokenizer(question, return_attention_mask=False)["input_ids"]
        mask_len = min(len(q_tokens), len(labels))
        labels[:mask_len] = -100
        labels[input_ids == self.processor.tokenizer.pad_token_id] = -100
        
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': input_ids,
            'labels': labels,
            'attention_mask': inputs['attention_mask'].squeeze(0)
        }

# Création Datasets
train_dataset = FacialExpressionDataset(train_df, processor, file_mapping, IMG_DIR)
eval_dataset = FacialExpressionDataset(test_df.head(50), processor, file_mapping, IMG_DIR)

# --- 4. TRAINER (3 EPOCHS) ---
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'input_ids': torch.stack([x['input_ids'] for x in batch]),
        'labels': torch.stack([x['labels'] for x in batch]),
        'attention_mask': torch.stack([x['attention_mask'] for x in batch])
    }

args = TrainingArguments(
    output_dir="/kaggle/working/blip2_final_3epochs",
    num_train_epochs=3,              # ✅ 3 EPOCHS
    per_device_train_batch_size=8,   # Batch Size
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collate_fn
)

print("\n🚀 C'est parti pour 3 EPOCHS (~6-8 min)...")
trainer.train()

# Sauvegarde
model.save_pretrained("/kaggle/working/saved_model")
processor.save_pretrained("/kaggle/working/saved_model")
print("\n✅ FINI ! Le modèle est entraîné et sauvegardé.")

🚀 DÉMARRAGE COMPLET (Tout-en-un)
📂 Mapping des fichiers...
📊 Chargement des CSV...
🤖 Chargement du modèle BLIP-2...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1598: UserWarning: The following device_map keys do not match any submodules in the model: ['query_tokens']
  warnings.warn(


trainable params: 5,242,880 || all params: 3,750,004,736 || trainable%: 0.1398

🚀 C'est parti pour 3 EPOCHS (~6-8 min)...


Step,Training Loss
10,16.918700
20,6.856400
30,2.540200
40,0.678200
50,0.336800
60,0.208000
70,0.174000
80,0.138400
90,0.133600
100,0.128200



✅ FINI ! Le modèle est entraîné et sauvegardé.


In [13]:
import torch
from tqdm import tqdm

print("📊 CALCUL DE L'ACCURACY FINALE")
print("-" * 30)

model.eval()
correct = 0
total = 0
target_count = 100 # On teste sur 100 images pour aller vite

# Vider le cache GPU pour éviter les erreurs
torch.cuda.empty_cache()

print(f"🔍 Test sur {target_count} images...")

for i in tqdm(range(len(test_df))):
    if total >= target_count:
        break
        
    try:
        item = test_df.iloc[i]
        real_file = file_mapping.get(item['image'])
        
        if real_file:
            img_path = os.path.join(IMG_DIR, real_file)
            image = Image.open(img_path).convert('RGB')
            
            # On pose la question
            prompt = "Question: What is the facial expression? Answer:"
            inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device, torch.float16)
            
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=10)
                prediction = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
            
            true_label = EMOTION_MAP[item['label']]
            
            # On vérifie si c'est bon
            if true_label.lower() in prediction.lower():
                correct += 1
            
            total += 1
    except:
        continue

accuracy = (correct / total) * 100
print("-" * 30)
print(f"🏆 RÉSULTAT FINAL : {accuracy:.2f}%")

if accuracy > 60:
    print("✅ SUCCÈS ! Ton modèle est performant.")
elif accuracy > 40:
    print("⚠️ PAS MAL. Le modèle a appris, mais peut faire mieux (plus d'epochs ?).")
else:
    print("❌ PROBLÈME. Le modèle n'a pas assez appris.")

📊 CALCUL DE L'ACCURACY FINALE
------------------------------
🔍 Test sur 100 images...


 11%|█         | 100/931 [00:50<06:58,  1.99it/s]

------------------------------
🏆 RÉSULTAT FINAL : 55.00%
⚠️ PAS MAL. Le modèle a appris, mais peut faire mieux (plus d'epochs ?).
